In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import random
import json
import time
import re

In [ ]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
gsm_train = load_dataset("openai/gsm8k", "main", split="train")
arc_train = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="train")

In [ ]:
random.seed(42)
gsm_idx = random.sample(range(len(gsm8k)), 100)
arc_idx = random.sample(range(len(arc)), 100)
print(len(gsm8k), len(arc))

In [ ]:
def generate(model,tokenizer, user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

In [ ]:
def gsm_prompt(q, mode):
        return f"Solve the problems step by step. At the end write 'Answer: <number>'.\n\nExamples:{gsm_shots[mode]}\n\nQuestion: {q}"

def arc_prompt(q, options, mode):
    opts = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(options))
    return f"Think step by step. At the end write 'Answer: <letter>'.\n\n{arc_shots[mode]}\n\nQuestion: {q}\n{opts}\nReasoning:"

def extract_number(text):
    m = re.search(r"Answer:\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        s = m.group(1)
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
        s = nums[-1] if nums else None
    return s.replace(",", "").rstrip(".") if s else None
    
def extract_letter(text):
    m = re.search(r"Answer:\s*\(?([A-E])\b", text)
    if m:
        return m.group(1)
    m = re.search(r"\b([A-E])\b", text)
    return m.group(1) if m else None
    
def same_number(a, b):
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return False

def split_steps(text):
    lines = [s.strip() for s in text.split("\n\n") if s.strip()]
    return [s for s in lines if not s.startswith("Answer:")]
    

In [ ]:
def gsm_example(solution,i):
    ex = gsm_train[i]
    final = ex["answer"].split("####")[1].strip()
    return f"Question: {ex['question']}\n{solution}\nAnswer: {final.strip()}"
    
    
def arc_example(ex, reasoning):
    opts = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(ex["choices"]["text"]))
    gold = chr(65 + ex["choices"]["label"].index(ex["answerKey"]))
    return f"Question: {ex['question']}\n{opts}\nReasoning: {reasoning}\nAnswer: {gold}"

In [ ]:
gsm_short = [
    "48/2 = 24; 48+24 = 72.",
    "12/60 = 0.2; 0.2*50 = 10.",
    "100/2 = 50; 15*2 = 30; 100-50-30-15 = 5.",
]
gsm_long = [
    "In April Natalia sold 48 clips. In May she sold half as many as in April, so we divide 48 by 2 and get 24 clips. To find how many clips she sold altogether in April and May, we add the two amounts: 48 + 24 = 72 clips.",
    "Weng earns $12 for every hour, and an hour has 60 minutes, so she earns 12 / 60 = $0.2 for every minute. She worked 50 minutes, so we multiply the earnings per minute by 50: 0.2 x 50 = $10.",
    "The wallet costs $100 and Betty has only half of that, so she has 100 / 2 = $50. Her parents give her $15. Her grandparents give her twice as much as her parents, which is 15 * 2 = $30. Together she now has 50 + 15 + 30 = $95. She still needs 100 - 95 = $5 more.",
]
arc_reasonings = [
    "Rubbing creates heat through friction, and more friction produces more heat. Dry palms have the most friction, while water, oil and lotion make the skin slippery and reduce friction.",
    "Magnets are attracted to certain metals, especially iron. A refrigerator door contains iron, so a magnet sticks to it, while smoothness, conductivity or electric wires do not cause magnetic attraction.",
    "Folds in sedimentary rock layers form when the layers are squeezed and bent by large forces. Converging crustal plates push against each other and create this pressure, while cooling magma, river deposits and dissolving minerals do not bend rock layers.",
    "The most recent scientific explanation is that a huge asteroid hit the Earth and its impact created dust that blocked the sunlight. Without sunlight plants could not grow and many animals died, while disease, mountain building and mammals preying on other animals are not the accepted explanation.",
    "The river current pushes the boat north, but the boat actually travels northeast, which is a combination of north and east. So the wind must provide the eastward push on the sails.",
]

arc_short = [
    "Dry palms have the most friction.",
    "Magnets stick to iron, and the door contains iron.",
    "Converging plates squeeze and bend rock layers.",
]
arc_long = [
    "Rubbing creates heat through friction, and more friction produces more heat. Dry palms rub against each other with the most friction. Wet palms, palms covered with oil and palms covered with lotion are slippery, so they have less friction and produce less heat. Therefore dry palms produce the most heat.",
    "Magnets are attracted to certain metals, especially iron. A refrigerator door contains iron, so a magnet sticks to it. A smooth surface does not attract a magnet, being a good conductor does not make a material magnetic, and electric wires in the door have nothing to do with magnetic attraction. Therefore the iron in the door is the best explanation.",
    "Folds in layers of sedimentary rock form when the layers are squeezed and bent by very large forces. Converging crustal plates push against each other and create exactly this kind of pressure. Cooling magma does not bend sedimentary layers, deposition of river sediments only adds new layers, and dissolving carbonate minerals does not bend layers either. Therefore the converging of crustal plates is the most likely cause.",
]

In [ ]:
gsm_shots = {
    "fewshot_k1":    "\n\n".join(gsm_example(gsm_train[i]['answer'],i) for i in range(1)),   
    "fewshot_k5":    "\n\n".join(gsm_example(gsm_train[i]['answer'],i) for i in range(5)),  
    "fewshot_short": "\n\n".join(gsm_example(gsm_short[i],i)for i in range(3)),                                         
    "fewshot_long":  "\n\n".join(gsm_example(gsm_long[i],i)for i in range(3)),                                          
}
arc_shots = {
    "fewshot_k1":    "\n\n".join(arc_example(arc_train[i], arc_reasonings[i]) for i in range(1)),
    "fewshot_k5":    "\n\n".join(arc_example(arc_train[i], arc_reasonings[i]) for i in range(5)),
    "fewshot_short": "\n\n".join(arc_example(arc_train[i], arc_short[i]) for i in range(3)),
    "fewshot_long":  "\n\n".join(arc_example(arc_train[i], arc_long[i]) for i in range(3)),
}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
model.eval()

In [ ]:
rows = []
count=0
for i in gsm_idx:
    count+=1
    if count%10==0: print(count)
    ex = gsm8k[i]
    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    for mode in ["fewshot_short", "fewshot_long", "fewshot_k1", "fewshot_k5"]:
        p = gsm_prompt(ex["question"], mode)
        t0 = time.time()
        txt, n = generate(model,tokenizer,p, 700)
        t = time.time() - t0
        pred = extract_number(txt)
        rows.append(dict(question_id=f"gsm8k_{i}", dataset="gsm8k", model="Qwen/Qwen2.5-0.5B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=same_number(pred, gold), n_tokens=n,gen_time_s=t))
count=0
for i in arc_idx:
    count+=1
    if count%10==0: print(count)
    ex = arc[i]
    labels = ex["choices"]["label"]
    gold = chr(65 + labels.index(ex["answerKey"]))
    for mode in ["fewshot_short", "fewshot_long", "fewshot_k1", "fewshot_k5"]:
        p = arc_prompt(ex["question"], ex["choices"]["text"], mode)
        t0 = time.time()
        txt, n = generate(model,tokenizer, p, 700)
        t = time.time() - t0
        pred = extract_letter(txt)
        rows.append(dict(question_id=f"arc_{i}", dataset="arc", model="Qwen/Qwen2.5-0.5B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=(pred == gold), n_tokens=n,gen_time_s=t))

df = pd.DataFrame(rows)
df["truncated"] = (df["mode"] != "direct") & (df["n_tokens"] >= 700)
df["steps"] = df.apply(lambda r: json.dumps(split_steps(r["generated_text"]) if r["mode"] != "direct" else [], ensure_ascii=False), axis=1)
df["n_steps"] = df["steps"].apply(lambda s: len(json.loads(s)))
df.to_csv("/kaggle/working/results_test100_Qwen2.5-0.5B-Instruct_fewshot.csv", index=False)
df

In [ ]:
tokenizer_2 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
model_2 = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", device_map="auto")
model_2.eval()

In [ ]:
rows = []
count=0
for i in gsm_idx:
    count+=1
    if count%10==0: print(count)
    ex = gsm8k[i]
    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    for mode in ["fewshot_short", "fewshot_long", "fewshot_k1", "fewshot_k5"]:
        p = gsm_prompt(ex["question"], mode)
        t0 = time.time()
        txt, n = generate(model_2,tokenizer_2,p, 700)
        t = time.time() - t0
        pred = extract_number(txt)
        rows.append(dict(question_id=f"gsm8k_{i}", dataset="gsm8k", model="Qwen/Qwen2.5-3B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=same_number(pred, gold), n_tokens=n,gen_time_s=t))
count=0
for i in arc_idx:
    count+=1
    if count%10==0: print(count)
    ex = arc[i]
    labels = ex["choices"]["label"]
    gold = chr(65 + labels.index(ex["answerKey"]))
    for mode in ["fewshot_short", "fewshot_long", "fewshot_k1", "fewshot_k5"]:
        p = arc_prompt(ex["question"], ex["choices"]["text"], mode)
        t0 = time.time()
        txt, n = generate(model_2,tokenizer_2, p, 700)
        t = time.time() - t0
        pred = extract_letter(txt)
        rows.append(dict(question_id=f"arc_{i}", dataset="arc", model="Qwen/Qwen2.5-3B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=(pred == gold), n_tokens=n,gen_time_s=t))

df_2 = pd.DataFrame(rows)
df_2["truncated"] = (df_2["mode"] != "direct") & (df_2["n_tokens"] >= 700)
df_2["steps"] = df_2.apply(lambda r: json.dumps(split_steps(r["generated_text"]) if r["mode"] != "direct" else [], ensure_ascii=False), axis=1)
df_2["n_steps"] = df_2["steps"].apply(lambda s: len(json.loads(s)))
df_2.to_csv("/kaggle/working/results_test100_Qwen2.5-0.3B-Instruct_fewshot.csv", index=False)
df_2

In [ ]:
tokenizer_3 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model_3 = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", device_map="auto")
model_3.eval()

In [ ]:
rows = []
count=0
for i in gsm_idx:
    count+=1
    if count%10==0: print(count)
    ex = gsm8k[i]
    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    for mode in ["fewshot_short", "fewshot_long", "fewshot_k1", "fewshot_k5"]:
        p = gsm_prompt(ex["question"], mode)
        t0 = time.time()
        txt, n = generate(model_3,tokenizer_3,p, 700)
        t = time.time() - t0
        pred = extract_number(txt)
        rows.append(dict(question_id=f"gsm8k_{i}", dataset="gsm8k", model="Qwen/Qwen2.5-7B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=same_number(pred, gold), n_tokens=n,gen_time_s=t))
count=0
for i in arc_idx:
    count+=1
    if count%10==0: print(count)
    ex = arc[i]
    labels = ex["choices"]["label"]
    gold = chr(65 + labels.index(ex["answerKey"]))
    for mode in ["fewshot_short", "fewshot_long", "fewshot_k1", "fewshot_k5"]:
        p = arc_prompt(ex["question"], ex["choices"]["text"], mode)
        t0 = time.time()
        txt, n = generate(model_3,tokenizer_3, p, 700)
        t = time.time() - t0
        pred = extract_letter(txt)
        rows.append(dict(question_id=f"arc_{i}", dataset="arc", model="Qwen/Qwen2.5-7B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=(pred == gold), n_tokens=n,gen_time_s=t))

df3 = pd.DataFrame(rows)

In [ ]:
df3["truncated"] = (df3["mode"] != "direct") & (df3["n_tokens"] >= 700)
df3["steps"] = df3.apply(lambda r: json.dumps(split_steps(r["generated_text"]) if r["mode"] != "direct" else [], ensure_ascii=False), axis=1)
df3["n_steps"] = df3["steps"].apply(lambda s: len(json.loads(s)))
df3.to_csv("/kaggle/working/results_test100_Qwen2.5-0.7B-Instruct_fewshot.csv", index=False)
df3